## Repair Misbound Authorships — oxjob #608 (0-REPAIR, one-time batch)

Heals historical positional author-id misbinding: `author_id` is bound to
`(work_id, author_sequence)` while author arrays shift, leaving ~9.9M bound rows whose name is
incompatible with the bound profile. This notebook detects them, resolves a repair per seat
(rebind to the right existing profile, or NULL — wrong attribution is worse than missing),
and applies in precision-gated waves. **Run manually, PROD only, in the quiet window between
end2end runs** (no concurrent UpdateWorkAuthors / ApplyWorkAuthorCurations / MatchAuthors).

Safety contract (the invariants from the 4 guard review rounds — see oxjob #608 REPAIR.md):
- **Never mints.** Repair assigns existing profile ids or NULL; nothing here creates authors.
- **Never fights curations.** Claim-curated seats are exempt (the Apply layer re-asserts
  name-anchored claims every cycle); curator-removed ids are never rebind candidates.
- **Live revalidation.** The detector snapshot is a hypothesis: every seat is re-judged
  against current `work_authors` state at resolve time, and the MERGE only touches seats
  still holding the id seen at resolve time.
- **Work-level exclusivity.** A rebind id lands on at most one seat per work and never
  collides with a seat that keeps its binding; any ambiguity resolves to NULL.
- **CJK / unparsed abstain.** Frozen-parser classes are cohort-tagged and never auto-acted on.
- **Apply is double-gated**: `repair_apply` defaults FALSE (resolve + gate are read-only),
  and every apply statement requires it.

Run protocol, per wave:
1. Set `repair_wave` (+ `batch_max_works`, keep `repair_apply` FALSE) → run Steps 0–2.
2. Run Step 3 gate; review the sample (ai_query judge + manual skim). **Proceed only ≥98%
   correct-or-harmless.**
3. Confirm end2end is idle, set `repair_apply` TRUE, re-run the Step 2 cells (fresh live
   state), then run Step 4 cells in order. Set `repair_apply` back to FALSE.
4. Run Step 5 verification.

| wave | cohort | apply_nulls | notes |
|---|---|---|---|
| `wave0_corpus` | actionable rows on the 4,997-row verification-corpus works | false | pilot; acceptance = verify_fix flips to RESOLVED |
| `wave1_legacy_strict` | legacy era, strict cross-surname | false | rebinds only; NULLs deferred |
| `wave1_legacy_strict` (rerun) | same | true | strict remainder → NULL |
| `wave2_legacy_loose` | legacy era, loose flags | false→true | own gate; may be rejected entirely |
| `wave3_new_era` | new-era flags | false→true | run LAST — NULLed rows re-enter the organic matcher |
| `wave4_no_profile` | dangling author_ids | true | no corroboration needed |


In [ ]:
DECLARE OR REPLACE VARIABLE repair_wave STRING DEFAULT 'wave0_corpus';

In [ ]:
-- Master apply gate. FALSE = resolve/gate only (read-only). Flip to TRUE just for Step 4,
-- after the precision gate passes and end2end is confirmed idle; flip back after.
DECLARE OR REPLACE VARIABLE repair_apply BOOLEAN DEFAULT false;

In [ ]:
-- Whether NULL resolutions (no safe rebind candidate) are applied this run, or deferred.
DECLARE OR REPLACE VARIABLE apply_nulls BOOLEAN DEFAULT false;

In [ ]:
-- Throttle: max works admitted per batch. Start small (wave 0 ignores this in practice;
-- wave 1 starts at 50K works), grow ~4x per clean batch.
DECLARE OR REPLACE VARIABLE batch_max_works BIGINT DEFAULT 50000;

### Step 0: Durable bookkeeping

`oxjob608_repair_log` is the append-only audit + rollback record: one row per applied change
(old id → new id). It also drives idempotence — seats already applied are skipped on re-runs.
Reversible row-by-row; never RESTORE mid-pipeline tables.

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.oxjob608_repair_log (
  applied_at TIMESTAMP,
  wave STRING,
  work_id BIGINT,
  author_sequence INT,
  raw_author_name STRING,
  old_author_id BIGINT,
  new_author_id BIGINT,
  rebind_tier STRING
)

### Step 1: DETECT — one-time full-scan snapshot (run ONCE)

Judges every bound `work_authors` row (name vs bound profile) with the v2 predicate and
tags each non-compatible row with a cohort. **Expensive (~820M bound rows, 4 joins) — run on
a large warehouse.** Both statements are `CREATE TABLE IF NOT EXISTS`, so re-running the
notebook later is a no-op; to re-snapshot deliberately, DROP both tables first.

Cohorts: `legacy_strict` (provable cross-surname) / `legacy_loose` / `new_era` /
`no_profile` (dangling id) — actionable; `isolated` / `abstain_cjk` / `abstain_unparsed` /
`curated_hold` — recorded, never auto-acted on.

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.oxjob608_repair_candidates_raw AS
WITH bound AS (
    SELECT work_id, author_sequence, raw_author_name, author_id,
           work_id > 7000000000 AS new_era
    FROM openalex.works.work_authors
    WHERE author_id IS NOT NULL
      AND raw_author_name IS NOT NULL AND TRIM(raw_author_name) != ''
),
keyed AS (
    SELECT b.*,
        COALESCE(oa.display_name, ar.display_name) AS profile_name,
        (oa.id IS NULL AND ar.id IS NULL) AS no_profile,
        an_r.match_last AS r_last, an_r.match_first AS r_first,
        -- profile keys: display_name row, falling back to full_name keys (curated display
        -- names can be absent from author_names — the guard round-4 donor-fallback fix)
        CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_last ELSE an_pf.match_last END AS p_last,
        CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_first ELSE an_pf.match_first END AS p_first
    FROM bound b
    LEFT JOIN openalex.authors.openalex_authors oa ON b.author_id = oa.id
    LEFT JOIN openalex.authors.authors ar ON b.author_id = ar.id
    LEFT JOIN openalex.authors.author_names an_r ON TRIM(b.raw_author_name) = an_r.raw_author_name
    LEFT JOIN openalex.authors.author_names an_p
        ON TRIM(COALESCE(oa.display_name, ar.display_name)) = an_p.raw_author_name
    LEFT JOIN openalex.authors.author_names an_pf
        ON TRIM(oa.full_name) = an_pf.raw_author_name
),
judged AS (
    SELECT *,
        CASE
            WHEN no_profile THEN 'NO_PROFILE'
            WHEN raw_author_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]'
              OR profile_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]' THEN 'ABSTAIN_CJK'
            WHEN r_last IS NULL OR p_last IS NULL THEN
                CASE WHEN LOWER(TRIM(raw_author_name)) = LOWER(TRIM(profile_name))
                     THEN 'COMPATIBLE' ELSE 'ABSTAIN_UNPARSED' END
            WHEN openalex.authors.names_compatible(
                     r_last, r_first, p_last, p_first,
                     raw_author_name, profile_name) THEN 'COMPATIBLE'
            ELSE 'INCOMPATIBLE'
        END AS verdict
    FROM keyed
),
-- COMPATIBLE rows contribute nothing to the count, so filter before the window
retained AS (
    SELECT work_id, author_sequence, raw_author_name, author_id, new_era,
           profile_name, r_last, r_first, p_last, p_first, verdict
    FROM judged WHERE verdict != 'COMPATIBLE'
)
SELECT *,
    COUNT(CASE WHEN verdict = 'INCOMPATIBLE' THEN 1 END)
        OVER (PARTITION BY work_id) AS work_incompat_count
FROM retained;

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.oxjob608_repair_candidates AS
WITH raw AS (
    SELECT * FROM openalex.authors.oxjob608_repair_candidates_raw
),
isolated_flag_works AS (
    -- displacement evidence is only ever USED for single-flag works (rows with
    -- work_incompat_count >= 2 pass corroboration on their own), so compute it only
    -- there — this also eliminates a quadratic name-comparison blowup on
    -- hyperauthorship works (2,300 flagged seats x 2,300 seats each)
    SELECT work_id FROM raw WHERE verdict = 'INCOMPATIBLE'
    GROUP BY work_id HAVING MAX(work_incompat_count) = 1
),
seats AS (
    SELECT wa.work_id, wa.author_sequence, wa.raw_author_name,
           an.match_last, an.match_first
    FROM openalex.works.work_authors wa
    JOIN isolated_flag_works fw ON wa.work_id = fw.work_id
    LEFT JOIN openalex.authors.author_names an ON TRIM(wa.raw_author_name) = an.raw_author_name
    WHERE wa.raw_author_name IS NOT NULL
),
displaced AS (
    -- displacement evidence: the flagged seat's profile person provably sits at ANOTHER
    -- seat of the same work (their name matches a different position's raw name)
    SELECT DISTINCT r.work_id, r.author_sequence
    FROM raw r
    JOIN seats s ON s.work_id = r.work_id AND s.author_sequence != r.author_sequence
    WHERE r.verdict = 'INCOMPATIBLE' AND r.work_incompat_count = 1
      AND r.p_last IS NOT NULL AND s.match_last IS NOT NULL
      AND openalex.authors.names_compatible(
              s.match_last, s.match_first, r.p_last, r.p_first,
              s.raw_author_name, r.profile_name)
),
curated AS (
    -- name-branch only (a claim on this seat's name): the id-branch was a round-3 defect
    SELECT DISTINCT r.work_id, r.author_sequence
    FROM raw r
    JOIN openalex.works.work_author_claim_curations cc
        ON cc.work_id = r.work_id
       AND LOWER(TRIM(cc.raw_author_name)) = LOWER(TRIM(r.raw_author_name))
),
enriched AS (
    SELECT r.*,
        (d.work_id IS NOT NULL) AS displaced,
        (cu.work_id IS NOT NULL) AS curated_hold,
        (r.verdict = 'INCOMPATIBLE' AND r.r_last IS NOT NULL AND r.p_last IS NOT NULL
         AND levenshtein(r.r_last, r.p_last) > 2
         AND instr(r.r_last, r.p_last) = 0 AND instr(r.p_last, r.r_last) = 0) AS strict
    FROM raw r
    LEFT JOIN displaced d ON r.work_id = d.work_id AND r.author_sequence = d.author_sequence
    LEFT JOIN curated cu ON r.work_id = cu.work_id AND r.author_sequence = cu.author_sequence
)
SELECT *,
    CASE
        WHEN verdict = 'NO_PROFILE' THEN 'no_profile'
        WHEN verdict = 'ABSTAIN_CJK' THEN 'abstain_cjk'
        WHEN verdict = 'ABSTAIN_UNPARSED' THEN 'abstain_unparsed'
        WHEN curated_hold THEN 'curated_hold'
        WHEN work_incompat_count < 2 AND NOT displaced THEN 'isolated'
        WHEN new_era THEN 'new_era'
        WHEN strict THEN 'legacy_strict'
        ELSE 'legacy_loose'
    END AS cohort,
    current_timestamp() AS detected_at
FROM enriched;

In [ ]:
-- Cohort sizing readout: the real wave sizes (SIM2 was a 0.1% row sample; distinct-work
-- counts do not extrapolate linearly). Sanity vs predictions: ~8.0M legacy + ~1.8M new
-- incompatible, ~1.7M no_profile.
SELECT cohort, new_era, COUNT(*) AS rows, COUNT(DISTINCT work_id) AS works
FROM openalex.authors.oxjob608_repair_candidates
GROUP BY cohort, new_era
ORDER BY rows DESC;

### Step 2: RESOLVE — admit a wave and recompute the repair against LIVE state

Work-level admission (exclusivity and realignment need every admitted seat of a work
processed together), then per seat: re-judge the CURRENT name vs the CURRENT profile
(curations/matcher may have healed it since the snapshot — those rows drop out), re-check
claim curations, re-corroborate, then the proven cascade:
**freed-donor realign → provenance-gated legacy roster → NULL**, with receiver exclusivity,
occupied-seat collision checks, and curator-removed ids excluded.

`rebind_tier = 'keep'` marks seats where the legacy roster confirms the CURRENT id for the
current name (contamination-renamed profile — the id is right, the profile name is wrong);
these are never touched and resolve later as CreateAuthors deflates the profile.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.oxjob608_wave_works AS
SELECT work_id
FROM (
    SELECT DISTINCT c.work_id
    FROM openalex.authors.oxjob608_repair_candidates c
    WHERE (repair_wave = 'wave0_corpus'
           AND c.cohort IN ('legacy_strict', 'legacy_loose', 'new_era')
           AND c.work_id IN (SELECT work_id
                             FROM openalex_dev.authors.misbinding_verification_corpus_20260714))
       OR (repair_wave = 'wave1_legacy_strict' AND c.cohort = 'legacy_strict')
       OR (repair_wave = 'wave2_legacy_loose'  AND c.cohort = 'legacy_loose')
       OR (repair_wave = 'wave3_new_era'       AND c.cohort = 'new_era')
       OR (repair_wave = 'wave4_no_profile'    AND c.cohort = 'no_profile')
)
QUALIFY ROW_NUMBER() OVER (ORDER BY work_id) <= batch_max_works;

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.oxjob608_repair_batch AS
WITH admitted AS (
    -- wave rows on admitted works, minus seats a previous batch already applied
    SELECT c.work_id, c.author_sequence, c.author_id AS snapshot_author_id,
           c.cohort, c.new_era, c.displaced
    FROM openalex.authors.oxjob608_repair_candidates c
    JOIN openalex.authors.oxjob608_wave_works w ON c.work_id = w.work_id
    WHERE ((repair_wave = 'wave0_corpus'        AND c.cohort IN ('legacy_strict', 'legacy_loose', 'new_era'))
        OR (repair_wave = 'wave1_legacy_strict' AND c.cohort = 'legacy_strict')
        OR (repair_wave = 'wave2_legacy_loose'  AND c.cohort = 'legacy_loose')
        OR (repair_wave = 'wave3_new_era'       AND c.cohort = 'new_era')
        OR (repair_wave = 'wave4_no_profile'    AND c.cohort = 'no_profile'))
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_repair_log g
                      WHERE g.work_id = c.work_id AND g.author_sequence = c.author_sequence)
),
live AS (
    -- LIVE revalidation: the snapshot is a hypothesis; judge the seat as it exists NOW
    SELECT a.work_id, a.author_sequence, a.cohort, a.new_era, a.displaced,
           a.snapshot_author_id,
           wa.raw_author_name AS live_name, wa.author_id AS live_author_id
    FROM admitted a
    JOIN openalex.works.work_authors wa
        ON wa.work_id = a.work_id AND wa.author_sequence = a.author_sequence
    WHERE wa.author_id IS NOT NULL
      AND wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) != ''
),
rejudged AS (
    SELECT l.*,
        COALESCE(oa.display_name, ar.display_name) AS old_profile_name,
        an_r.match_last AS r_last, an_r.match_first AS r_first,
        CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_last ELSE an_pf.match_last END AS p_last,
        CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_first ELSE an_pf.match_first END AS p_first,
        CASE
            WHEN oa.id IS NULL AND ar.id IS NULL THEN 'NO_PROFILE'
            WHEN l.live_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]'
              OR COALESCE(oa.display_name, ar.display_name) RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]' THEN 'ABSTAIN_CJK'
            WHEN an_r.match_last IS NULL
              OR (an_p.match_last IS NULL AND an_pf.match_last IS NULL) THEN
                CASE WHEN LOWER(TRIM(l.live_name)) = LOWER(TRIM(COALESCE(oa.display_name, ar.display_name)))
                     THEN 'COMPATIBLE' ELSE 'ABSTAIN_UNPARSED' END
            WHEN openalex.authors.names_compatible(
                     an_r.match_last, an_r.match_first,
                     CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_last ELSE an_pf.match_last END,
                     CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_first ELSE an_pf.match_first END,
                     l.live_name, COALESCE(oa.display_name, ar.display_name)) THEN 'COMPATIBLE'
            ELSE 'INCOMPATIBLE'
        END AS live_verdict
    FROM live l
    LEFT JOIN openalex.authors.openalex_authors oa ON l.live_author_id = oa.id
    LEFT JOIN openalex.authors.authors ar ON l.live_author_id = ar.id
    LEFT JOIN openalex.authors.author_names an_r ON TRIM(l.live_name) = an_r.raw_author_name
    LEFT JOIN openalex.authors.author_names an_p
        ON TRIM(COALESCE(oa.display_name, ar.display_name)) = an_p.raw_author_name
    LEFT JOIN openalex.authors.author_names an_pf
        ON TRIM(oa.full_name) = an_pf.raw_author_name
),
still AS (
    -- act only on seats that are STILL bad live, and not (re)claimed by a curation since
    SELECT r.* FROM rejudged r
    WHERE (r.live_verdict = 'INCOMPATIBLE'
           OR (r.live_verdict = 'NO_PROFILE' AND r.cohort = 'no_profile'))
      AND NOT EXISTS (SELECT 1 FROM openalex.works.work_author_claim_curations cc
                      WHERE cc.work_id = r.work_id
                        AND LOWER(TRIM(cc.raw_author_name)) = LOWER(TRIM(r.live_name)))
),
corroborated AS (
    SELECT *,
        COUNT(CASE WHEN live_verdict = 'INCOMPATIBLE' THEN 1 END)
            OVER (PARTITION BY work_id) AS live_incompat_count
    FROM still
),
eligible AS (
    -- corroboration re-checked live; no_profile is a dangling reference (none needed)
    SELECT * FROM corroborated
    WHERE (live_verdict = 'NO_PROFILE' AND cohort = 'no_profile')
       OR (live_verdict = 'INCOMPATIBLE' AND (live_incompat_count >= 2 OR displaced))
),
freed AS (
    -- ids this wave frees on the work, with their profile identity as the donor name
    SELECT e.work_id, e.author_sequence AS freed_seq, e.live_author_id AS donor_id,
           e.old_profile_name AS donor_name, e.p_last AS donor_last, e.p_first AS donor_first
    FROM eligible e
    WHERE e.old_profile_name IS NOT NULL
),
realign_cand AS (
    SELECT e.work_id, e.author_sequence,
        COUNT(DISTINCT CASE WHEN LOWER(TRIM(f.donor_name)) = LOWER(TRIM(e.live_name))
                            THEN f.donor_id END) AS n_exact,
        MIN(CASE WHEN LOWER(TRIM(f.donor_name)) = LOWER(TRIM(e.live_name))
                 THEN f.donor_id END) AS id_exact,
        COUNT(DISTINCT CASE WHEN openalex.authors.names_compatible(
                                 e.r_last, e.r_first, f.donor_last, f.donor_first,
                                 e.live_name, f.donor_name)
                            THEN f.donor_id END) AS n_compat,
        MIN(CASE WHEN openalex.authors.names_compatible(
                      e.r_last, e.r_first, f.donor_last, f.donor_first,
                      e.live_name, f.donor_name)
                 THEN f.donor_id END) AS id_compat
    FROM eligible e
    JOIN freed f ON f.work_id = e.work_id AND f.freed_seq != e.author_sequence
    GROUP BY e.work_id, e.author_sequence
),
realign_chosen AS (
    SELECT work_id, author_sequence,
        CASE WHEN n_exact = 1 THEN id_exact
             WHEN n_exact = 0 AND n_compat = 1 THEN id_compat
        END AS realign_id
    FROM realign_cand
),
legacy_roster AS (
    SELECT l.work_id, l.author_id, l.raw_author_name
    FROM openalex.works_legacy.work_authors l
    JOIN openalex.authors.oxjob608_wave_works w ON l.work_id = w.work_id
    WHERE l.author_id IS NOT NULL
),
provenance AS (
    -- legacy tier only when the seat's CURRENT id came from the legacy roster
    -- (blocks reverting curation reassignments the roster never contained).
    -- Uses the UNFILTERED roster: a dangling current id (no_profile wave) still
    -- proves legacy carry-over provenance.
    SELECT DISTINCT e.work_id, e.author_sequence
    FROM eligible e
    JOIN legacy_roster lr ON lr.work_id = e.work_id AND lr.author_id = e.live_author_id
),
legacy_targets AS (
    -- rebind TARGETS must still exist as profiles: the roster remembers ids merged
    -- away since, and rebinding to a ghost recreates the no_profile class (first
    -- observation run: 308 of 23,970 event hypotheses pointed at dead ids)
    SELECT lr.*
    FROM legacy_roster lr
    LEFT JOIN openalex.authors.openalex_authors pe ON lr.author_id = pe.id
    LEFT JOIN openalex.authors.authors ae ON lr.author_id = ae.id
    WHERE pe.id IS NOT NULL OR ae.id IS NOT NULL
),
legacy_exact AS (
    SELECT e.work_id, e.author_sequence,
           MIN(lr.author_id) AS id_l, COUNT(DISTINCT lr.author_id) AS n_l
    FROM eligible e
    JOIN legacy_targets lr
        ON lr.work_id = e.work_id
       AND LOWER(TRIM(lr.raw_author_name)) = LOWER(TRIM(e.live_name))
    GROUP BY e.work_id, e.author_sequence
),
legacy_parsed AS (
    SELECT e.work_id, e.author_sequence,
           MIN(lr.author_id) AS id_l, COUNT(DISTINCT lr.author_id) AS n_l
    FROM eligible e
    JOIN legacy_targets lr ON lr.work_id = e.work_id
    JOIN openalex.authors.author_names pn ON TRIM(lr.raw_author_name) = pn.raw_author_name
    WHERE e.r_last IS NOT NULL
      AND pn.match_last = e.r_last
      AND COALESCE(pn.match_first, '') = COALESCE(e.r_first, '')
    GROUP BY e.work_id, e.author_sequence
),
cascade AS (
    SELECT e.*,
        rc.realign_id,
        CASE WHEN p.work_id IS NOT NULL THEN
            CASE WHEN le.n_l = 1 THEN le.id_l
                 WHEN le.work_id IS NULL AND lp.n_l = 1 THEN lp.id_l
            END
        END AS legacy_id
    FROM eligible e
    LEFT JOIN realign_chosen rc
        ON e.work_id = rc.work_id AND e.author_sequence = rc.author_sequence
    LEFT JOIN provenance p
        ON e.work_id = p.work_id AND e.author_sequence = p.author_sequence
    LEFT JOIN legacy_exact le
        ON e.work_id = le.work_id AND e.author_sequence = le.author_sequence
    LEFT JOIN legacy_parsed lp
        ON e.work_id = lp.work_id AND e.author_sequence = lp.author_sequence
),
candidate AS (
    SELECT *, COALESCE(realign_id, legacy_id) AS rebind_candidate
    FROM cascade
),
occupied AS (
    -- ids that will REMAIN bound on the work after this wave (seats not being repaired)
    SELECT DISTINCT wa.work_id, wa.author_id
    FROM openalex.works.work_authors wa
    JOIN openalex.authors.oxjob608_wave_works w ON wa.work_id = w.work_id
    LEFT ANTI JOIN eligible e
        ON wa.work_id = e.work_id AND wa.author_sequence = e.author_sequence
    WHERE wa.author_id IS NOT NULL
),
removes AS (
    -- an author a curator explicitly removed from the work is never a valid candidate
    SELECT DISTINCT rc.work_id, rc.author_id
    FROM openalex.works.work_author_remove_curations rc
    JOIN openalex.authors.oxjob608_wave_works w ON rc.work_id = w.work_id
),
resolved AS (
    SELECT c.work_id, c.author_sequence, c.cohort, c.new_era,
        c.live_name, c.live_author_id, c.old_profile_name,
        c.r_last, c.r_first, c.realign_id, c.legacy_id,
        CASE WHEN c.rebind_candidate IS NULL THEN NULL
             WHEN o.author_id IS NOT NULL THEN NULL
             WHEN rm.author_id IS NOT NULL THEN NULL
             WHEN COUNT(*) OVER (PARTITION BY c.work_id, c.rebind_candidate) > 1 THEN NULL
             ELSE c.rebind_candidate
        END AS rebind_author_id
    FROM candidate c
    LEFT JOIN occupied o ON c.work_id = o.work_id AND c.rebind_candidate = o.author_id
    LEFT JOIN removes rm ON c.work_id = rm.work_id AND c.rebind_candidate = rm.author_id
)
SELECT *,
    CASE WHEN rebind_author_id IS NULL THEN 'null'
         WHEN rebind_author_id = live_author_id THEN 'keep'
         WHEN rebind_author_id = realign_id THEN 'realign'
         ELSE 'legacy'
    END AS rebind_tier,
    current_timestamp() AS resolved_at
FROM resolved;

In [ ]:
-- Batch readout. Expectations from SIM3/dry-run (legacy): ~91% rebindable, realign+legacy
-- split, 'keep' small (contamination-renamed profiles the id actually fits).
SELECT cohort, rebind_tier, COUNT(*) AS rows, COUNT(DISTINCT work_id) AS works
FROM openalex.authors.oxjob608_repair_batch
GROUP BY cohort, rebind_tier
ORDER BY cohort, rows DESC;

### Step 3: GATE — stratified precision sample before any apply

~70 rows per tier (realign / legacy / null; 'keep' rows are never applied so not sampled),
deterministic hash order. Judge with ai_query, then manually skim every disagreement and a
slice of the agreements. **Do not run Step 4 unless correct-or-harmless ≥98%.** Save the
judged sample to the oxjobs evidence directory for the wave record.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.oxjob608_gate_sample AS
SELECT b.*, COALESCE(oa.display_name, ar.display_name) AS new_profile_name
FROM openalex.authors.oxjob608_repair_batch b
LEFT JOIN openalex.authors.openalex_authors oa ON b.rebind_author_id = oa.id
LEFT JOIN openalex.authors.authors ar ON b.rebind_author_id = ar.id
WHERE b.rebind_tier != 'keep'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY b.rebind_tier
    ORDER BY ABS(HASH(b.work_id, b.author_sequence))
) <= 70;

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.oxjob608_gate_judged AS
SELECT s.*,
    ai_query(
        'databricks-claude-opus-4-8',
        'You are auditing a repair of author-name to author-profile bindings on scholarly works. '
        || 'A position on a work has an author name string. The OLD profile bound there was judged wrong '
        || '(surname family does not match). The repair proposes a NEW binding: an existing profile, or NULL '
        || 'meaning unbind and leave the position unattributed. '
        || 'Author name at the position: "' || s.live_name || '". '
        || 'OLD profile display name being removed: "' || COALESCE(s.old_profile_name, '(dangling id, no profile)') || '". '
        || 'NEW binding: ' || CASE WHEN s.rebind_author_id IS NULL THEN 'NULL (unbind)'
                                   ELSE '"' || COALESCE(s.new_profile_name, '(profile without name)') || '"' END || '. '
        || 'Answer whether this change is correct-or-harmless. It is correct when the NEW profile name plausibly '
        || 'denotes the same person as the author name at the position, or when the new binding is NULL and the '
        || 'OLD profile was clearly a different person. It is harmful when the OLD profile was actually the right '
        || 'person, or the NEW profile is a different person than the author name.',
        responseFormat => '{"type": "json_schema", "json_schema": {"name": "verdict", "schema": {"type": "object", "properties": {"correct_or_harmless": {"type": "boolean"}, "reason": {"type": "string"}}, "required": ["correct_or_harmless", "reason"]}, "strict": true}}'
    ) AS judge_json
FROM openalex.authors.oxjob608_gate_sample s;

In [ ]:
-- Gate score per tier + overall. Manually review every not-ok row before deciding.
SELECT rebind_tier,
    COUNT(*) AS judged,
    COUNT(CASE WHEN CAST(judge_json:correct_or_harmless AS BOOLEAN) THEN 1 END) AS ok,
    ROUND(100.0 * COUNT(CASE WHEN CAST(judge_json:correct_or_harmless AS BOOLEAN) THEN 1 END)
          / COUNT(*), 2) AS pct_ok
FROM openalex.authors.oxjob608_gate_judged
GROUP BY ROLLUP(rebind_tier)
ORDER BY rebind_tier;

### Step 4: APPLY — gated on `repair_apply` (and `apply_nulls` for NULL resolutions)

Pre-apply checklist — ALL of these, every batch:
1. Gate passed ≥98% for this wave (Step 3, this batch).
2. end2end is IDLE (check the walden_end2end job) — no concurrent author-side MERGE.
3. Step 2 was re-run just now if any time passed since the gate (fresh live state).
4. `repair_apply` set TRUE for exactly these three cells; back to FALSE after.

Order matters: log (intent record) → MERGE → pending_sync. The MERGE's MATCHED condition
re-checks the seat still holds the id seen at resolve time — anything that moved since is
skipped (compare the log vs live counts in the last cell of this step). 'keep' rows are
never applied.

In [ ]:
INSERT INTO openalex.authors.oxjob608_repair_log
SELECT current_timestamp() AS applied_at,
       repair_wave AS wave,
       b.work_id, b.author_sequence,
       b.live_name AS raw_author_name,
       b.live_author_id AS old_author_id,
       b.rebind_author_id AS new_author_id,
       b.rebind_tier
FROM openalex.authors.oxjob608_repair_batch b
JOIN openalex.works.work_authors wa
    ON wa.work_id = b.work_id AND wa.author_sequence = b.author_sequence
   AND wa.author_id <=> b.live_author_id
WHERE repair_apply
  AND b.rebind_tier != 'keep'
  AND (b.rebind_author_id IS NOT NULL OR apply_nulls);

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT work_id, author_sequence, live_author_id, rebind_author_id
    FROM openalex.authors.oxjob608_repair_batch
    WHERE repair_apply
      AND rebind_tier != 'keep'
      AND (rebind_author_id IS NOT NULL OR apply_nulls)
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id <=> source.live_author_id THEN
    UPDATE SET target.author_id = source.rebind_author_id,
               target.updated_at = current_timestamp();

In [ ]:
INSERT INTO openalex.works.curated_work_ids_pending_sync (work_id, added_datetime)
SELECT DISTINCT b.work_id, current_timestamp() AS added_datetime
FROM openalex.authors.oxjob608_repair_batch b
WHERE repair_apply
  AND b.rebind_tier != 'keep'
  AND (b.rebind_author_id IS NOT NULL OR apply_nulls)
  AND NOT EXISTS (SELECT 1 FROM openalex.works.curated_work_ids_pending_sync pending
                  WHERE pending.work_id = b.work_id);

In [ ]:
-- Post-apply check on the batch just applied (latest applied_at = one statement = one
-- timestamp): logged intents vs live state. diverged > 0 means seats moved between log and
-- MERGE (quiet-window violation) — investigate before the next batch.
SELECT
    COUNT(*) AS logged,
    COUNT(CASE WHEN wa.author_id <=> g.new_author_id THEN 1 END) AS applied_live,
    COUNT(CASE WHEN NOT (wa.author_id <=> g.new_author_id) THEN 1 END) AS diverged
FROM openalex.authors.oxjob608_repair_log g
LEFT JOIN openalex.works.work_authors wa
    ON wa.work_id = g.work_id AND wa.author_sequence = g.author_sequence
WHERE g.applied_at = (SELECT MAX(applied_at) FROM openalex.authors.oxjob608_repair_log);

### Step 5: VERIFY + sanity-check against the observation guard

The first cells compare the detector snapshot against `author_guard_events` (the shipped
observation layer) — the planned pre-wave-1 sanity check. Notes for reading the crosstab:
the guard judges name *transitions* on works end2end touches, the detector judges resting
*state* of every bound row — overlap is partial by design. Seats with a guard event but no
detector row were either corrupted after the snapshot or judged compatible at snapshot time;
both are expected. Then verify_fix against the corpus, and the wave audit.

In [ ]:
SELECT * FROM openalex.authors.author_guard_telemetry ORDER BY run_at DESC LIMIT 10;

In [ ]:
-- Guard event mix so far. Sanity vs SIM2: incompatible ~1.1-1.8% of changed positions;
-- isolated-dominated works ~45% of flagged works.
SELECT verdict,
    COUNT(*) AS events,
    COUNT(CASE WHEN invalidate THEN 1 END) AS would_invalidate,
    COUNT(CASE WHEN rebind_author_id IS NOT NULL THEN 1 END) AS with_rebind_hypothesis,
    COUNT(DISTINCT work_id) AS works
FROM openalex.authors.author_guard_events
GROUP BY verdict
ORDER BY events DESC;

In [ ]:
-- Detector cohort × guard verdict on overlapping seats (NULL cohort = not in snapshot:
-- post-snapshot corruption or compatible-at-snapshot; expected).
SELECT e.verdict AS guard_verdict, c.cohort AS detector_cohort,
    COUNT(*) AS seats, COUNT(DISTINCT e.work_id) AS works
FROM openalex.authors.author_guard_events e
LEFT JOIN openalex.authors.oxjob608_repair_candidates c
    ON c.work_id = e.work_id AND c.author_sequence = e.author_sequence
GROUP BY e.verdict, c.cohort
ORDER BY seats DESC;

In [ ]:
-- What happened to guard would-invalidate seats since capture: how many the pipeline
-- already healed (matches hypothesis / changed otherwise) vs still standing. Informs how
-- much wave-5 (events top-up) work actually accumulates.
SELECT
    (e.rebind_author_id IS NOT NULL) AS has_rebind_hypothesis,
    CASE WHEN wa.work_id IS NULL THEN 'seat_gone'
         WHEN wa.author_id <=> e.postmerge_author_id THEN 'unchanged_since_capture'
         WHEN wa.author_id <=> e.rebind_author_id THEN 'now_matches_hypothesis'
         ELSE 'changed_other'
    END AS live_state,
    COUNT(*) AS seats
FROM openalex.authors.author_guard_events e
LEFT JOIN openalex.works.work_authors wa
    ON wa.work_id = e.work_id AND wa.author_sequence = e.author_sequence
WHERE e.invalidate
GROUP BY 1, 2
ORDER BY seats DESC;

In [ ]:
-- verify_fix (inline from oxjob #608 evidence/sql/verify_fix.sql): corpus resolution status.
-- Baseline 99.5-100% STILL_MISBOUND; wave-0 target ~0 immediate + RESOLVED_profile_fixed
-- trailing in over subsequent CreateAuthors runs.
WITH fold AS (
  SELECT
    c.*,
    wa.author_id AS current_author_id,
    wa.raw_author_name AS current_raw_name,
    a.display_name AS current_profile_name,
    lower(regexp_replace(
      CASE WHEN instr(wa.raw_author_name, ',') > 0 THEN trim(split(wa.raw_author_name, ',')[0])
           ELSE element_at(split(trim(wa.raw_author_name), ' '), -1) END,
      '[^a-zA-Z]', '')) AS raw_last,
    lower(regexp_replace(
      CASE WHEN instr(a.display_name, ',') > 0 THEN trim(split(a.display_name, ',')[0])
           ELSE element_at(split(trim(a.display_name), ' '), -1) END,
      '[^a-zA-Z]', '')) AS prof_last,
    lower(regexp_replace(wa.raw_author_name, '[^a-zA-Z]', '')) AS raw_flat,
    lower(regexp_replace(a.display_name,     '[^a-zA-Z]', '')) AS prof_flat
  FROM openalex_dev.authors.misbinding_verification_corpus_20260714 c
  LEFT JOIN openalex.works.work_authors wa
    ON c.work_id = wa.work_id AND c.author_sequence = wa.author_sequence
  LEFT JOIN openalex.authors.openalex_authors a
    ON wa.author_id = a.id
),
judged AS (
  SELECT
    era,
    CASE
      WHEN current_author_id IS NULL THEN 'RESOLVED_row_removed'
      WHEN current_raw_name <> raw_author_name THEN 'REJUDGE_raw_name_changed'
      WHEN raw_last = prof_last
        OR levenshtein(raw_last, prof_last) <= 2
        OR instr(prof_flat, raw_last) > 0
        OR instr(raw_flat, prof_last) > 0 THEN
        CASE WHEN current_author_id <> author_id
             THEN 'RESOLVED_reassigned' ELSE 'RESOLVED_profile_fixed' END
      ELSE 'STILL_MISBOUND'
    END AS status
  FROM fold
)
SELECT era, status, COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY era), 2) AS pct_of_era
FROM judged
GROUP BY era, status
ORDER BY era, status;

In [ ]:
-- Wave audit: everything ever applied, by wave/day/tier.
SELECT wave, DATE(applied_at) AS day, rebind_tier,
    COUNT(*) AS rows, COUNT(DISTINCT work_id) AS works
FROM openalex.authors.oxjob608_repair_log
GROUP BY wave, DATE(applied_at), rebind_tier
ORDER BY day, wave, rebind_tier;